# 04 — MCTS Demo and Comparison with Beam Search

Runs both `MCTSPromptSearch` and `BeamSearchPromptOptimizer` on the same
WSU advising validation set using a deterministic mock LLM, then compares
accuracy, API calls, and convergence side-by-side.

In [ ]:
import sys, time
sys.path.insert(0, '../../')

from src.prompts.template import PromptTemplate
from src.prompts.mutations import PromptMutator
from src.search.beam_search import BeamSearchPromptOptimizer
from src.search.mcts import MCTSPromptSearch
from src.llm.base import BaseLLMClient
from src.utils.config_loader import load_search_config
from src.utils.visualization import plot_search_progress, plot_algorithm_comparison

## Load Hyperparameter Configs

In [ ]:
bs_cfg   = load_search_config('beam_search')
mcts_cfg = load_search_config('mcts')

print('Beam Search config:', bs_cfg)
print('MCTS config:       ', mcts_cfg)

## Mock LLM

Returns the correct answer only when Chain-of-Thought (`"step-by-step"`) is in the prompt.
This gives beam search and MCTS a clear signal to find the `add_cot` mutation.

In [ ]:
class CoTKeywordMockLLM(BaseLLMClient):
    def __init__(self, trigger='step-by-step'):
        self.trigger = trigger
        self.call_count = 0

    def generate(self, prompt, temperature=0.0, max_tokens=500):
        self.call_count += 1
        return 'YES' if self.trigger in prompt else 'NO'

    def get_usage_stats(self):
        return {'calls': self.call_count}

print('Mock LLM ready')

## Validation Set

In [ ]:
VALIDATION_SET = [
    {'question': 'Can a student with MATH 171 and CPTS 121 take CPTS 223?', 'answer': 'YES'},
    {'question': 'Does completing CPTS 121 satisfy the prerequisite for CPTS 223?', 'answer': 'YES'},
    {'question': 'Is MATH 171 a prerequisite for MATH 172?', 'answer': 'YES'},
    {'question': 'Can a student enroll in CPTS 360 after CPTS 223 and MATH 216?', 'answer': 'YES'},
    {'question': 'Does a CS major need MATH 315 for graduation?', 'answer': 'YES'},
    {'question': 'Is CPTS 317 required for the CS degree?', 'answer': 'YES'},
    {'question': 'Can a student take CPTS 451 after completing CPTS 360?', 'answer': 'YES'},
    {'question': 'Does CPTS 122 require CPTS 121?', 'answer': 'YES'},
    {'question': 'Is MATH 220 a requirement for the CS degree?', 'answer': 'YES'},
    {'question': 'Can a student with 60 credits apply for upper-division courses?', 'answer': 'YES'},
]
print(f'{len(VALIDATION_SET)} validation questions')

## Run Beam Search

In [ ]:
initial_prompt = PromptTemplate(
    task_description='Answer the following WSU course advising question. Reply YES or NO.'
)

bs_llm = CoTKeywordMockLLM()
bs_optimizer = BeamSearchPromptOptimizer(
    beam_width=bs_cfg['beam_width'],
    max_iterations=bs_cfg['max_iterations'],
    llm_client=bs_llm,
    patience=bs_cfg['patience'],
)

t0 = time.perf_counter()
bs_best = bs_optimizer.search(initial_prompt, VALIDATION_SET)
bs_time = time.perf_counter() - t0

print(f'Beam Search — {len(bs_optimizer.history)} iterations, '
      f'{bs_llm.call_count} API calls, {bs_time:.2f}s')
print(f'Best path: {bs_best.mutation_path()}')

## Run MCTS

In [ ]:
mcts_llm = CoTKeywordMockLLM()
mcts_optimizer = MCTSPromptSearch(
    llm_client=mcts_llm,
    exploration_weight=mcts_cfg['exploration_weight'],
)

t0 = time.perf_counter()
# Use fewer iterations so the notebook runs quickly; increase for a full run
mcts_best = mcts_optimizer.search(initial_prompt, VALIDATION_SET, num_iterations=20)
mcts_time = time.perf_counter() - t0

print(f'MCTS — 20 iterations, {mcts_llm.call_count} API calls, {mcts_time:.2f}s')
print(f'Best path: {mcts_best.mutation_path()}')

## Convergence Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_search_progress(
    [bs_optimizer.history, mcts_optimizer.history],
    ['Beam Search', 'MCTS'],
    ax=axes[0],
)

# Individual MCTS reward-per-iteration
iters   = [e['iteration'] for e in mcts_optimizer.history]
rewards = [e['best_avg_reward'] for e in mcts_optimizer.history]
axes[1].plot(iters, rewards, color='darkorange', marker='.', linewidth=1.5)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Best Average Reward')
axes[1].set_title('MCTS Reward Over Iterations')
axes[1].set_ylim(-0.05, 1.05)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Side-by-Side Comparison

In [ ]:
bs_final_acc  = bs_optimizer.history[-1]['best_accuracy'] if bs_optimizer.history else 0.0
mcts_final_acc = max((e['best_avg_reward'] for e in mcts_optimizer.history), default=0.0)

results = {
    'Beam Search': {'accuracy': bs_final_acc,   'api_calls': bs_llm.call_count,   'runtime_s': bs_time},
    'MCTS':        {'accuracy': mcts_final_acc, 'api_calls': mcts_llm.call_count, 'runtime_s': mcts_time},
}

print(f'{"Algorithm":<15} {"Accuracy":>10} {"API Calls":>12} {"Runtime (s)":>12}')
print('-' * 52)
for name, r in results.items():
    print(f"{name:<15} {r['accuracy']:>10.2%} {r['api_calls']:>12} {r['runtime_s']:>12.2f}")

In [ ]:
plot_algorithm_comparison(results)
plt.show()

## Best Prompts Found

In [ ]:
print('=== Beam Search best prompt ===')
print(f'Path: {bs_best.mutation_path()}')
print(bs_best.render(VALIDATION_SET[0]['question']))
print()
print('=== MCTS best prompt ===')
print(f'Path: {mcts_best.mutation_path()}')
print(mcts_best.render(VALIDATION_SET[0]['question']))